# 1. EDA and Schema Inference for Fitness Data

This notebook performs an initial exploratory data analysis (EDA) on the fitness dataset. The goal is to understand the data's structure, infer a schema, and identify any potential data quality issues.

Since the sample Excel file cannot be accessed directly, this notebook assumes the data is in a CSV format at the specified S3 location. We will proceed based on a typical schema for fitness tracking data.

**Objectives:**
1.  Set up a Spark session.
2.  Define a schema and load the data from S3.
3.  Inspect the data and its schema.
4.  Calculate summary statistics.
5.  Perform a null value analysis.
6.  Explore distributions of key columns.


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql.functions import col, count, when, isnan

# Set up Spark Session
# In a real Glue/SageMaker environment, the Spark session is often pre-configured.
spark = (
    SparkSession.builder.appName("Fitness-EDA")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") # Add AWS S3 support
    .getOrCreate()
)

print("Spark session created successfully.")


# 2. Load Data from S3

Next, we will define the schema for our fitness data and load the CSV file from the specified S3 path. Defining a schema upfront is more robust and performant than relying on schema inference, as it prevents incorrect data types and handles potential inconsistencies in the data.

We will use the S3 path `s3://my-bucket/raw/fitness.csv` as requested.
_Note: Ensure the environment you run this in has the correct AWS credentials and permissions to access this S3 path._


In [ ]:
# S3 path for the raw fitness data
s3_path = "s3://my-bucket/raw/fitness.csv"

# Define the schema based on typical fitness data
fitness_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("activity_type", StringType(), True),
    StructField("duration_minutes", DoubleType(), True),
    StructField("calories_burned", DoubleType(), True),
    StructField("steps", IntegerType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("heart_rate", IntegerType(), True),
    StructField("sleep_hours", DoubleType(), True),
    StructField("stress_level", IntegerType(), True),
    StructField("weight_kg", DoubleType(), True),
])

# Read the CSV data from S3
try:
    df = spark.read.csv(
        s3_path,
        schema=fitness_schema,
        header=True,
        timestampFormat="yyyy-MM-dd'T'HH:mm:ss" # Adjust format as needed
    )
    print(f"Successfully loaded data from {s3_path}")
except Exception as e:
    print(f"Failed to load data from S3. Error: {e}")
    # Create an empty dataframe with the schema to allow the rest of the notebook to run
    df = spark.createDataFrame([], schema=fitness_schema)


# 3. Schema and Data Inspection

Let's start by inspecting the schema to ensure all data types were read correctly. We will also display a few rows of the DataFrame to get a feel for the data.


In [ ]:
# Print the DataFrame schema
print("DataFrame Schema:")
df.printSchema()

# Show the first 5 rows of the DataFrame
print("Sample Data:")
df.show(5, truncate=False)


# 4. Summary Statistics

The `describe()` function provides a quick overview of the numerical columns in the DataFrame, including count, mean, standard deviation, min, and max values. This is a crucial step for identifying outliers or anomalies.


In [ ]:
# Calculate and show summary statistics for numerical columns
print("Summary Statistics:")
df.describe().show()


# 5. Null Value Analysis

Checking for null or missing values is a critical part of data cleaning. We will count the number of nulls in each column to understand the completeness of our dataset.


In [ ]:
# Count null values for each column
print("Null Value Counts:")
df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns]).show()


# 6. Basic EDA

Let's perform some basic analysis on our categorical and numerical columns.

### Activity Type Distribution

Understanding the distribution of different activities can provide insights into user behavior.


In [ ]:
# Show the distribution of the 'activity_type' column
print("Distribution of Activity Types:")
df.groupBy("activity_type").count().orderBy(col("count").desc()).show()


# 7. Conclusion & Next Steps

This initial EDA has provided a foundational understanding of the fitness dataset. We have:
-   Defined a robust schema.
-   Inspected the data for correctness.
-   Calculated summary statistics to spot potential outliers.
-   Checked for missing values.
-   Analyzed the distribution of activity types.

**Next Steps:**
-   Based on these findings, the Glue ETL job (`glue_jobs/etl.py`) can be refined with specific data cleaning logic.
-   More advanced visualizations can be created to explore relationships between variables (e.g., heart rate vs. activity type).
-   The feature engineering steps in the Glue job can be implemented based on insights gathered here.
